# Ludwig AutoML benchmark

Runs Ludwig's AutoML engine (`ludwig.automl.auto_train`) on the 14 datasets in the project's
`datasets/` folder, with several seeds per dataset. Ludwig receives the whole dataset with all
its defaults, plus two settings: the seed and the 20% test share. It splits the data itself
(70% train / 10% validation / 20% test) and the reported metrics are those of its test split.
No cross-validation; results are averaged over the seeds.

**To run it:** check the *Parameters* cell below (run mode, time limit, datasets, targets and
seeds) and *Run All*. Everything a reviewer may want to change is in that cell; the rest of
the notebook does not need editing.

Requires: `requirements.txt` (Ludwig 0.17.9).


## Parameters

The only cell to edit.

In [ ]:
import os

# ---- Run mode ------------------------------------------------------------------------------
# True  -> smoke test: only TEST_DATASET, its first seed, 300 s (about 7 minutes)
# False -> the full experiment: every dataset in DATASETS, all its seeds, 1000 s per run
TEST_MODE = True
TEST_DATASET = "diabetes"  # the dataset used when TEST_MODE is True

# ---- Ludwig --------------------------------------------------------------------------------
TIME_LIMIT_S = 300 if TEST_MODE else 1000  # time budget of each run, in seconds

# Shares of test and validation, as fractions of the WHOLE dataset; the rest is train. Ludwig's own
# defaults are 0.2 and 0.1 (so 0.7 train). Any values are accepted as long as the three are positive.
TEST_FRACTION = 0.2
VALIDATION_FRACTION = 0.1

# ---- Datasets, targets and seeds -----------------------------------------------------------
# name: (folder inside datasets/, target column, seeds). The folder gives the task type
# (Binary and Classification are classification, Regression is regression). The target is stated
# because auto_train requires it and cannot detect it. A dataset is run once per seed, and a
# seed drives Ludwig's split, its hyperparameter sampling and its training.
DATASETS = {
    "blood-transfusion-service-center": ("Binary", "Class", [17, 30, 56, 476, 777]),
    "breast-w": ("Binary", "Class", [17, 30, 56, 131]),
    "credit-g": ("Binary", "class", [17, 19, 30, 56, 131, 300926]),
    "diabetes": ("Binary", "class", [4, 23, 56, 131, 300926]),
    "qsar-biodeg": ("Binary", "Class", [4, 21, 131, 4444, 300926]),
    "analcatdata_dmft": ("Classification", "Prevention", [4, 17, 21, 300926]),
    "cmc": ("Classification", "Contraceptive_method_used", [4, 21, 468]),
    "hypothyroid": ("Classification", "Class", [468]),
    "mfeat-morphological": ("Classification", "class", [17, 468]),
    "vehicle": ("Classification", "Class", [17, 42, 112]),
    "cholesterol": ("Regression", "chol", [17, 42, 50]),
    "cloud": ("Regression", "TE", [42, 52, 78]),
    "liver-disorders": ("Regression", "drinks", [42, 87, 100, 476]),
    "plasma_retinol": ("Regression", "RETPLASMA", [30, 100, 476, 10343]),
}

# ---- Interruptions -------------------------------------------------------------------------
# RESUME = True : a dataset with all its runs saved is skipped; one that is only partly done is
#                 discarded and restarted from its first seed. False: start over (the previous
#                 results of this run mode are moved aside, not deleted).
# DATASET_RETRIES: extra attempts at a dataset (from its first seed) when one of its runs fails.
RESUME = True
DATASET_RETRIES = 1

# ---- Paths (the notebook runs with its own folder, benchmarks/ludwig, as working directory) ---
DATASETS_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "datasets"))
RESULTS_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "results"))


In [ ]:
# Derived from the parameters above; nothing to edit here.
if TEST_DATASET not in DATASETS:
    raise ValueError(f"TEST_DATASET '{TEST_DATASET}' is not in DATASETS.")
if not (TEST_FRACTION > 0 and VALIDATION_FRACTION > 0 and TEST_FRACTION + VALIDATION_FRACTION < 1):
    raise ValueError("TEST_FRACTION and VALIDATION_FRACTION must be positive and leave some data for training.")

# results/<RUN_MODE_NAME>/ ; the test share is in the name so that results with another split never mix
RUN_MODE_NAME = f"{'test' if TEST_MODE else 'full'}_{TIME_LIMIT_S}s_test{round(TEST_FRACTION * 100)}pct"
DATASET_NAMES = [TEST_DATASET] if TEST_MODE else list(DATASETS)
# Seeds each dataset is run with (the first one only in test mode)
SEEDS_TO_RUN = {name: DATASETS[name][2][:1] if TEST_MODE else DATASETS[name][2] for name in DATASET_NAMES}

# Saved with the results (parameters.json) so the analysis and the reader know the settings
PARAMETERS = {
    "run_mode": RUN_MODE_NAME,
    "test_mode": TEST_MODE,
    "time_limit_s": TIME_LIMIT_S,
    "test_fraction": TEST_FRACTION,
    "validation_fraction": VALIDATION_FRACTION,
    "resume": RESUME,
    "dataset_retries": DATASET_RETRIES,
    "datasets_dir": DATASETS_DIR,
    "datasets": {name: list(DATASETS[name][:2]) for name in DATASET_NAMES},
    "seeds_per_dataset": SEEDS_TO_RUN,
}
print(f"Run mode '{RUN_MODE_NAME}': {len(DATASET_NAMES)} dataset(s), "
      f"{sum(len(s) for s in SEEDS_TO_RUN.values())} run(s) of up to {TIME_LIMIT_S} s each.")


In [ ]:
import sys

# common_utils.py lives in the parent folder
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import json
import time

from common_utils import disable_unsupported_gpu, cleanup_ludwig_artifacts, init_ray_for_ludwig, log_available_memory

# Must run before importing ludwig/torch (falls back to CPU on old GPUs)
disable_unsupported_gpu()

from ludwig.automl import auto_train
from ludwig.modules.metric_modules import get_best_function

from common_utils import (
    load_dataset,
    save_parameters,
    save_environment_info,
    load_completed_runs,
    start_fresh_if_requested,
    archive_dataset_results,
    safe_metric_value,
    save_incremental_results,
)


## Ludwig experiment functions

In [ ]:
def load_best_model_local(results):
    """Loads the best trial's model on the local backend.

    Ludwig's `results.best_model` calls LudwigModel.load() without a backend,
    so it auto-detects Ray (already running for hyperopt) and fails with
    "RayTrainerV2 has no attribute create_checkpoint_handle". This does the
    same reload but passes backend="local" explicitly.
    """
    from ludwig.api import LudwigModel
    from ludwig.globals import MODEL_FILE_NAME

    checkpoint = results.experiment_analysis.best_checkpoint
    if checkpoint is None:
        raise RuntimeError("No hyperopt trial produced a checkpoint")
    with checkpoint.as_directory() as ckpt_path:
        model_dir = os.path.join(ckpt_path, MODEL_FILE_NAME)
        if not os.path.isdir(model_dir):
            raise RuntimeError(f"Best checkpoint has no model files at {model_dir}")
        return LudwigModel.load(model_dir, backend="local", from_checkpoint=True)


# Share of train / validation / test that Ludwig is told to use (its defaults: 70 / 10 / 20)
SPLIT_PROBABILITIES = [round(1 - VALIDATION_FRACTION - TEST_FRACTION, 6), VALIDATION_FRACTION, TEST_FRACTION]


def train_ludwig(df, target_column, seed):
    """Ludwig AutoML (auto_train) on the WHOLE dataset; everything else is Ludwig's default.

    Ludwig splits the data itself into train / validation / test and keeps the test part
    out of training and model selection. Only the split shares (SPLIT_PROBABILITIES) and
    the seed are given to it. The split TYPE stays Ludwig's choice: stratified by the
    target when it is an imbalanced classification target, random otherwise.
    """
    return auto_train(
        dataset=df,
        target=target_column,  # auto_train cannot guess it: set per dataset in DATASETS (common_utils.py)
        time_limit_s=TIME_LIMIT_S,
        random_seed=seed,
        user_config={"preprocessing": {"split": {"probabilities": SPLIT_PROBABILITIES}}},
        # Ludwig silently switches to its distributed Ray backend when
        # "ray" is importable; that backend fails in Ludwig 0.17.x with
        # "RayTrainerV2 has no attribute create_checkpoint_handle".
        # Hyperopt still uses Ray Tune; only each trial trains locally.
        backend="local",
    )


def check_ludwig_split(model):
    """Fails unless the trained model is configured with the test share that was asked for (TEST_FRACTION)."""
    probabilities = ((model.config.get("preprocessing") or {}).get("split") or {}).get("probabilities")
    if probabilities is None or abs(probabilities[2] - TEST_FRACTION) > 1e-9:
        raise ValueError(f"Ludwig's split probabilities are {probabilities}, not the requested {TEST_FRACTION:.0%} test share.")


def read_ludwig_test_metrics(results, model, target_column):
    """Ludwig's metrics on its own held-out test split, for the model auto_train selected.

    Ludwig records, after every epoch of every trial, the metrics of the training,
    validation and test splits (`training_stats`). It picks the epoch with the best
    validation metric, so the test metrics of that same epoch are the ones to report
    (not the last epoch's). The test split plays no part in that choice.
    """
    best_trial = results.experiment_analysis.best_trial.last_result
    stats = json.loads(best_trial["training_stats"])

    metric = model.config["trainer"]["validation_metric"]  # what Ludwig selects the epoch by
    validation = stats["validation"][target_column][metric]
    best_epoch, best_value = get_best_function(metric)(enumerate(validation), key=lambda pair: pair[1])
    # Same number Ludwig ranks the trials by: proves we picked the epoch Ludwig picked.
    if abs(best_value - best_trial["metric_score"]) > 1e-6:
        raise ValueError(f"Best validation {metric} {best_value} != Ludwig's metric_score {best_trial['metric_score']}")

    return {m: values[best_epoch] for m, values in stats["test"][target_column].items()}


def extract_ludwig_diagnostics(results, model):
    """What Ludwig did during one run (trials, epochs, model, resources), for auditing.

    Answers questions such as "how many trials fitted in the time limit", "how
    long did the best one train" or "which model type was chosen". Every field
    is optional: a failure to read one never affects the metrics.
    """
    diag = {}
    try:
        analysis = results.experiment_analysis
        trials = list(analysis.trials)
        by_status = {}
        for t in trials:
            by_status[t.status] = by_status.get(t.status, 0) + 1
        best = analysis.best_trial
        last = best.last_result or {}
        diag.update({
            "n_trials": len(trials),
            "trials_by_status": by_status,
            "best_trial_id": best.trial_id,
            "best_trial_epochs": last.get("training_iteration"),
            "best_trial_time_s": last.get("time_total_s"),
            "sum_trial_time_s": round(sum(
                (t.last_result or {}).get("time_total_s", 0) for t in trials), 1),
            "best_hyperparameters": analysis.best_config,
        })
    except Exception as e:
        print(f"[WARN] Could not read Ludwig's trials: {e}")
    try:
        cfg = model.config
        trainer = cfg.get("trainer") or {}
        diag["model_type"] = cfg.get("model_type")
        diag["combiner"] = (cfg.get("combiner") or {}).get("type")
        diag["trainer"] = {k: trainer.get(k) for k in (
            "epochs", "batch_size", "learning_rate", "optimizer", "early_stop")}
    except Exception as e:
        print(f"[WARN] Could not read Ludwig's trainer settings: {e}")
    try:
        import ray
        if ray.is_initialized():  # still up here; cleanup_ludwig_artifacts() stops it later
            diag["ray_resources"] = ray.cluster_resources()
    except Exception as e:
        print(f"[WARN] Could not read Ray's resources: {e}")
    return diag


def run_ludwig_experiment(df, target_column, seed, task_type):
    """Trains one Ludwig AutoML run on the whole dataset: (metrics, feature types, diagnostics)."""
    cleanup_ludwig_artifacts()  # stale hyperopt/ from a previous run
    log_available_memory()
    init_ray_for_ludwig()  # GPUs shared with the desktop: see common_utils.py
    try:
        train_start = time.perf_counter()
        results = train_ludwig(df, target_column, seed)
        train_time = time.perf_counter() - train_start
        model = load_best_model_local(results)
        check_ludwig_split(model)
        check_output_type(model, target_column, task_type)
        test_metrics = read_ludwig_test_metrics(results, model, target_column)
        metrics = extract_ludwig_metrics(test_metrics)
        diagnostics = extract_ludwig_diagnostics(results, model)
        metrics["train_time_s"] = round(train_time, 2)  # auto_train only (time-limited)
        if "n_trials" in diagnostics:
            metrics["n_trials"] = diagnostics["n_trials"]
            metrics["n_trials_failed"] = diagnostics["trials_by_status"].get("ERROR", 0)

        # Logs what Ludwig decided on its own (feature types, encoders, combiner), for
        # reproducibility and auditing. Read from the loaded best model's config: Ray's
        # best_config only holds the sampled hyperparameters, not the full Ludwig config.
        inferred_types = None
        try:
            cfg = model.config
            inferred_types = {
                "input_features": {
                    f["name"]: {"type": f["type"], "encoder": (f.get("encoder") or {}).get("type")}
                    for f in cfg["input_features"]
                },
                "output_features": {f["name"]: f["type"] for f in cfg["output_features"]},
                "combiner": (cfg.get("combiner") or {}).get("type"),
            }
        except Exception as e:
            print(f"[WARN] Could not extract Ludwig's inferred feature types: {e}")

        del results
        return metrics, inferred_types, diagnostics
    except Exception as e:
        print(f"[ERROR] Ludwig, seed {seed}: {e}")
        return None, None, None
    finally:
        # Metrics are already in memory; trial dumps are no longer needed
        cleanup_ludwig_artifacts()


def check_output_type(model, target_column, task_type):
    """Fails if Ludwig's inferred type for the target is not the dataset's task (classification / regression)."""
    inferred = next(f["type"] for f in model.config["output_features"] if f["name"] == target_column)
    expected = ("category", "binary") if task_type == "classification" else ("number",)
    if inferred not in expected:
        raise ValueError(f"Ludwig inferred a '{inferred}' target for a {task_type} dataset.")


def extract_ludwig_metrics(test_metrics):
    """Every metric Ludwig computed on its test split for the target ({metric: value}).

    Nothing is filtered by task: classification gives e.g. accuracy, roc_auc, loss;
    regression gives e.g. r2, RMSE, MAE; whatever else Ludwig reports is kept too.
    """
    metrics = {m: safe_metric_value(v) for m, v in test_metrics.items()}
    return {m: v for m, v in metrics.items() if isinstance(v, (int, float))}


## Main loop

In [ ]:
# results/<run mode>/ludwig/ (next to analysis/ and logs/); see common_utils.py
save_dir = os.path.join(RESULTS_ROOT, RUN_MODE_NAME, "ludwig")
feature_type_log_path = os.path.join(save_dir, "ludwig_inferred_feature_types.json")
start_fresh_if_requested(save_dir, RESUME)  # only acts with RESUME = False
os.makedirs(save_dir, exist_ok=True)
save_parameters(save_dir, PARAMETERS)
save_environment_info(save_dir, PARAMETERS, RESUME)
diagnostics_log_path = os.path.join(save_dir, "ludwig_run_diagnostics.json")


def load_json_log(path):
    """Previous log, to keep what already-finished runs recorded when resuming."""
    if RESUME and os.path.isfile(path):
        try:
            with open(path) as f:
                return json.load(f)
        except Exception as e:
            print(f"[WARN] Could not read {os.path.basename(path)}: {e}")
    return {}


def save_json_log(path, log):
    try:
        with open(path, "w") as f:
            json.dump(log, f, indent=2, default=str)
    except Exception as e:
        print(f"[WARN] Could not save {os.path.basename(path)}: {e}")


feature_type_log = load_json_log(feature_type_log_path)
diagnostics_log = load_json_log(diagnostics_log_path)

for name in DATASET_NAMES:
    print(f"\n=== Dataset {name} ===")
    data = load_dataset(name, DATASETS, DATASETS_DIR)  # the whole dataset: Ludwig splits it itself
    if data is None:
        continue
    seeds = SEEDS_TO_RUN[name]  # one run per seed
    print(f"[INFO] {name}: target '{data['target_column']}', {len(data['df'])} rows, {len(seeds)} run(s) planned.")

    def forget_dataset_logs():
        """Drops what the discarded runs of this dataset recorded in the JSON logs."""
        for log in (feature_type_log, diagnostics_log):
            for key in [k for k in log if k.startswith(f"{name}_seed")]:
                del log[key]
        save_json_log(feature_type_log_path, feature_type_log)
        save_json_log(diagnostics_log_path, diagnostics_log)

    # A finished dataset is skipped; a half-done one is discarded and restarted from its first seed
    runs = load_completed_runs(save_dir, name, len(seeds), RESUME)
    if len(runs) < len(seeds):
        forget_dataset_logs()

    for attempt in range(1 + DATASET_RETRIES):
        if len(runs) == len(seeds):
            break
        if attempt > 0:
            print(f"[RETRY] {name}: a run failed; discarding this dataset's runs "
                  f"and starting it over (attempt {attempt + 1} of {1 + DATASET_RETRIES})")
            archive_dataset_results(save_dir, name)
            forget_dataset_logs()
            runs = []
        for seed in seeds:
            print(f"[RUN] {name} seed={seed}")
            run_start = time.perf_counter()
            metrics, inferred_types, diagnostics = run_ludwig_experiment(
                data["df"], data["target_column"], seed, data["task_type"],
            )
            run_time = time.perf_counter() - run_start
            if not metrics:
                break  # a failed run: the whole dataset is retried
            metrics["time_taken"] = round(run_time, 2)  # seconds, train + eval
            runs.append({"seed": seed, "metrics": metrics})
            run_key = f"{name}_seed{seed}"
            if inferred_types:
                feature_type_log[run_key] = inferred_types
            if diagnostics:
                diagnostics_log[run_key] = diagnostics
            # Saved after every run so an interruption loses nothing
            save_json_log(feature_type_log_path, feature_type_log)
            save_json_log(diagnostics_log_path, diagnostics_log)
            save_incremental_results(name, data["group"], runs, save_dir)
    if len(runs) < len(seeds):
        print(f"[WARN] {name}: only {len(runs)} of {len(seeds)} runs completed after "
              f"{1 + DATASET_RETRIES} attempt(s). Launch the benchmark again to restart it.")


print("\nAll datasets processed.")
print(f"Results in: {save_dir}")
print(f"Ludwig's inferred feature types: {feature_type_log_path}")
print(f"Ludwig's per-run diagnostics (trials, epochs, model): {diagnostics_log_path}")


## Analysis

Summary table and statistics from the results above (the same as `python analyze_results.py <mode>`). Edit `ANALYSIS_OUTPUTS` to choose which files are written.

In [ ]:
# Which analysis files to write; the summary table is always shown below.
# Available: summary_table, per_dataset_stats, time_and_failures, ludwig_diagnostics, datasets_summary.
ANALYSIS_OUTPUTS = ["summary_table", "per_dataset_stats", "time_and_failures", "ludwig_diagnostics",
                    "datasets_summary"]

try:
    from IPython.display import display
    from analyze_results import run_analysis  # lives in the parent folder

    summary = run_analysis(os.path.join(RESULTS_ROOT, RUN_MODE_NAME), outputs=ANALYSIS_OUTPUTS)
    display(summary)
except (Exception, SystemExit) as e:
    print(f"[WARN] The analysis could not run ({e}). From a terminal: "
          f"python analyze_results.py {RUN_MODE_NAME}")
